# Notebook 04: Plot-Level Surprisal

**Pipeline layer F.** Computes a discriminant-validity measure at the plot level rather than the element level. Reads `movies_union.parquet` and `stockiness_long_classified.parquet`. Produces per-film surprisal scores and the correlation between plot-level surprisal and per-plot mean element-stockiness.

## Purpose and doctrinal stakes

Notebooks 03 and 03b score elements one at a time within genres. The doctrinal claim those measures make is that they capture **element-level originality** — how stock a particular narrative element is within a particular genre. A competing skeptical hypothesis is that the element-level measures are just a re-skinned plot-predictability metric: films that read as conventional sequences of words are also films whose individual elements look stock under the framework's counts. If that hypothesis is right, the framework is not contributing anything beyond what a language model could compute by reading the plot directly.

The discriminant-validity test is the empirical resolution. For each film in the corpus, compute the plot-level surprisal under a long-context language model (per-token negative log probability averaged across the plot), and the per-plot mean of element-level stockiness (the average ensemble score across the narrative elements present in the film, averaged across the film's genres). Correlate the two at the corpus level and within each canonical genre.

The expected empirical signature is **modest correlation**: Pearson r below 0.4. A modest correlation means the two measures share a directional relationship (more-conventional plots contain more-stock elements, which is intuitive) but the relationship is loose enough that the element-level measures are capturing something the plot-level surprisal does not. The doctrinal claim survives. A high correlation (r above 0.6) would mean the element-level measures are essentially redundant with plot predictability, weakening the case that the framework operationalizes the doctrine independently.

## Method

Per-token surprisal under a long-context language model is the standard plot-predictability measure. For a plot with tokens t₁, t₂, ..., t_T:

$$s(t_i) = -\log P(t_i \mid t_1, t_2, \ldots, t_{i-1})$$

The plot-level surprisal is the per-token average over the full plot:

$$S(\text{plot}) = \frac{1}{T-1} \sum_{i=2}^{T} s(t_i)$$

The first token is excluded because it has no preceding context.

The model used is **Qwen2.5-7B**, a 7-billion-parameter long-context autoregressive language model with strong narrative-text perplexity scores. The model fits in A100 VRAM at float16 precision (≈14 GB occupied of 40 or 80 GB available). The maximum context window is well above the 99.9th-percentile plot length in our corpus (3,064 tokens), so no truncation logic is needed for the bulk of the corpus.

For each film, we also compute its per-plot mean stockiness:

$$\bar{S}_{\text{stock}}(f) = \text{mean}_{g \in G_f}\!\left(\text{mean}_{e \in f}\!\left[\text{ensemble}(e, g)\right]\right)$$

where G_f is the set of canonical genres the film f is tagged with, and the inner mean is over the narrative elements present in f (filtered to `is_narrative_element=True` from 03b). The double-mean is symmetric in elements and genres, which is appropriate because both vary across films.

The headline empirical quantity is the corpus-wide Pearson r between S(plot) and Ŝ_stock(f), with per-genre breakdowns reported alongside.

## Run protocol

The notebook has four phases. Phases B (the GPU surprisal computation) is the long one; everything else is light.

| Phase | What it does | Time | Resources |
|---|---|---|---|
| A | Load corpus and pipeline outputs | ~1 min | CPU |
| B | Per-plot surprisal under Qwen2.5-7B | 2–6 hours | A100 GPU |
| C | Compute per-film mean stockiness | ~5 min | CPU |
| D | Correlation analysis and report | ~1 min | CPU |

Total wall time: 2–6 hours, dominated by Phase B. No API spend. Requires Colab Pro+ with A100 access.

## Inputs

- `v4_rebuild/01c_union/movies_union.parquet` — 45,958 films with cleaned plot text
- `v4_rebuild/03b_classification/stockiness_long_classified.parquet` — per-element stockiness with narrative-element classification
- `v4_rebuild/02_elements/element_index.parquet` — per-film element membership

## Outputs

- `v4_rebuild/04_surprisal/per_film_surprisal.parquet` — one row per film, with `mean_surprisal`, `n_tokens`, `model_name`
- `v4_rebuild/04_surprisal/per_film_mean_stockiness.parquet` — one row per film, with `mean_stockiness_overall`, `n_elements`, plus per-genre means
- `v4_rebuild/04_surprisal/correlation_table.parquet` — corpus-wide and per-genre Pearson r and Spearman ρ
- `v4_rebuild/04_surprisal/surprisal_report.txt` — headline correlation plus per-genre breakdowns


In [1]:
# ----------------------------------------------------------------------
# Setup and configuration
# ----------------------------------------------------------------------
!pip install -q transformers accelerate torch

import os, time, math
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# --- Google Drive ---------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# --- Paths ---------------------------------------------------------------
ROOT = Path('/content/drive/MyDrive/DGK Lab/Reasoning Lab/'
            'DGK Lab - Scene A Faire Computational')
V4 = ROOT / 'Analysis/v4_rebuild'

PATH_MOVIES   = V4 / '01c_union'          / 'movies_union.parquet'
PATH_ENRICHED = V4 / '03b_classification' / 'stockiness_long_classified.parquet'
PATH_ELEM_IDX = V4 / '02_elements'        / 'element_index.parquet'

OUT_DIR = V4 / '04_surprisal'
OUT_DIR.mkdir(parents=True, exist_ok=True)

PATH_SURPRISAL  = OUT_DIR / 'per_film_surprisal.parquet'
PATH_MEAN_STOCK = OUT_DIR / 'per_film_mean_stockiness.parquet'
PATH_CORR_TBL   = OUT_DIR / 'correlation_table.parquet'
PATH_REPORT     = OUT_DIR / 'surprisal_report.txt'

# --- Run configuration ----------------------------------------------------
# Qwen2.5-7B is the primary model: strong narrative-text perplexity,
# fits A100 VRAM at float16, 32K context window so no plot needs truncating.
# Llama-3.1-8B is an alternative for sensitivity; uncomment to switch.
MODEL_NAME = 'Qwen/Qwen2.5-7B'
# MODEL_NAME = 'meta-llama/Llama-3.1-8B'

# Plots longer than this are truncated. The corpus's 99.9th percentile is
# 3,064 tokens, so the truncation almost never fires in practice; the cap
# guards against pathological outliers and keeps memory predictable.
MAX_TOKENS = 4096

# Save surprisal scores in batches of this size. Each batch flushes to the
# checkpoint parquet so a crashed run can resume.
CHECKPOINT_EVERY = 500

# Canonical genre list, identical to the rest of the pipeline.
CANONICAL_GENRES = [
    'Action', 'Adventure', 'Animation', 'Comedy', 'Crime',
    'Drama', 'Family', 'Fantasy', 'History', 'Horror',
    'Music', 'Mystery', 'Romance', 'Science Fiction',
    'Thriller', 'War', 'Western',
]

# Device check — A100 required for reasonable runtime.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda', (
    'No GPU detected. This notebook needs an A100 (Colab Pro+). '
    'Open Runtime > Change runtime type and select A100 GPU.'
)
print(f'Setup complete.')
print(f'  Model     : {MODEL_NAME}')
print(f'  Device    : {torch.cuda.get_device_name(0)}')
print(f'  Max tokens: {MAX_TOKENS}')
print(f'  Output dir: {OUT_DIR}')

Mounted at /content/drive
Setup complete.
  Model     : Qwen/Qwen2.5-7B
  Device    : NVIDIA A100-SXM4-80GB
  Max tokens: 4096
  Output dir: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/04_surprisal


In [2]:
# ----------------------------------------------------------------------
# Phase A — Load the corpus
# ----------------------------------------------------------------------
# The `plot` column of movies_union.parquet is already cleaned at the
# 01 audit stage (HTML entities decoded, Wikipedia templates and ref
# markers stripped). No further cleaning is needed before passing to
# the tokenizer.
# ----------------------------------------------------------------------
print('Loading movies_union.parquet...')
movies = pd.read_parquet(PATH_MOVIES)
movies['genres_canonical'] = movies['genres_canonical'].apply(
    lambda gs: list(gs) if hasattr(gs, '__len__') and not isinstance(gs, str) else []
)
print(f'  {len(movies):,} films')

# Drop films with empty or extremely short plot text (defensive — the audit
# should already have removed these, but the surprisal computation needs
# at least a handful of tokens to produce a meaningful average).
movies['plot_word_count'] = movies['plot'].fillna('').str.split().str.len()
keep = movies['plot_word_count'] >= 20
n_dropped = int((~keep).sum())
if n_dropped:
    print(f'  dropping {n_dropped:,} films with fewer than 20 plot words')
movies = movies[keep].reset_index(drop=True)
print(f'  {len(movies):,} films retained for surprisal computation')

# Quick distribution check on plot length
print(f'\n  plot word count percentiles: '
      f'p50={int(movies["plot_word_count"].median())}  '
      f'p90={int(movies["plot_word_count"].quantile(0.90))}  '
      f'p99={int(movies["plot_word_count"].quantile(0.99))}  '
      f'p99.9={int(movies["plot_word_count"].quantile(0.999))}  '
      f'max={int(movies["plot_word_count"].max())}')

Loading movies_union.parquet...
  45,958 films
  45,958 films retained for surprisal computation

  plot word count percentiles: p50=235  p90=772  p99=1412  p99.9=2463  max=5130


In [3]:
# ----------------------------------------------------------------------
# Load Qwen2.5-7B for next-token prediction.
#
# float16 keeps memory at ~14 GB; A100 has 40 or 80 GB, plenty of room.
# device_map='auto' lets accelerate place layers on the available GPU.
# We do not need a generation head, just the LM logits for surprisal.
# ----------------------------------------------------------------------
print(f'Loading {MODEL_NAME} (first-time download is ~14 GB, then cached)...')
t0 = time.time()

from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()

print(f'Model loaded in {time.time()-t0:.1f}s')
print(f'  vocab size  : {tokenizer.vocab_size:,}')
print(f'  context     : {getattr(model.config, "max_position_embeddings", "unknown")}')
print(f'  parameters  : ~{sum(p.numel() for p in model.parameters())/1e9:.1f}B')
print(f'  GPU memory  : {torch.cuda.memory_allocated()/1e9:.1f} GB allocated')

Loading Qwen/Qwen2.5-7B (first-time download is ~14 GB, then cached)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (3 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model loaded in 60.9s
  vocab size  : 151,643
  context     : 131072
  parameters  : ~7.6B
  GPU memory  : 15.2 GB allocated


In [4]:
# ----------------------------------------------------------------------
# Phase B — Per-plot surprisal under Qwen2.5-7B
#
# For each plot:
#   1. Tokenize (truncate to MAX_TOKENS if necessary).
#   2. Forward pass with `labels=input_ids` so the model returns
#      per-token cross-entropy, internally shifted so position i\'s loss
#      uses positions 0..i-1 as context. The model omits position 0 from
#      the loss (it has no context).
#   3. The returned `loss` is the mean cross-entropy = mean surprisal
#      across all valid positions. We multiply by (T-1) to recover the
#      total negative log probability, and store both mean and total.
#   4. Append the record to the in-memory list and flush to parquet
#      every CHECKPOINT_EVERY films.
#
# CHECKPOINT: PATH_SURPRISAL. Re-running this cell resumes from the
# last checkpoint, skipping films already processed.
# ----------------------------------------------------------------------

# Resume support: if the checkpoint parquet exists, identify which films
# are already done and skip them.
if PATH_SURPRISAL.exists():
    surprisal_so_far = pd.read_parquet(PATH_SURPRISAL)
    done_ids = set(surprisal_so_far['union_id'].astype(int))
    print(f'Resuming from checkpoint: {len(done_ids):,} films already done')
    records = surprisal_so_far.to_dict('records')
else:
    done_ids = set()
    records = []
    print('No checkpoint found; starting from scratch.')

remaining = movies[~movies['union_id'].isin(done_ids)].reset_index(drop=True)
print(f'{len(remaining):,} films to process')


@torch.no_grad()
def per_token_mean_surprisal(text):
    """Return (mean_surprisal, n_valid_tokens). n_valid_tokens is T-1 because
    the first token has no preceding context and is excluded from the loss."""
    enc = tokenizer(text, return_tensors='pt', truncation=True,
                    max_length=MAX_TOKENS)
    input_ids = enc.input_ids.to(DEVICE)
    if input_ids.shape[1] < 2:
        return float('nan'), 0
    out = model(input_ids, labels=input_ids)
    # out.loss is the mean cross-entropy over positions 1..T-1 (HuggingFace
    # shifts internally). Returned as natural-log nats per token.
    return float(out.loss.item()), int(input_ids.shape[1] - 1)


pbar = tqdm(remaining.iterrows(), total=len(remaining), desc='surprisal')
since_flush = 0
for _, row in pbar:
    try:
        mean_s, n_tok = per_token_mean_surprisal(row['plot'])
    except torch.cuda.OutOfMemoryError:
        # Fall back: truncate aggressively for this one row and retry
        torch.cuda.empty_cache()
        try:
            plot_short = ' '.join(row['plot'].split()[:1500])
            mean_s, n_tok = per_token_mean_surprisal(plot_short)
            print(f'  OOM on union_id={row["union_id"]}; truncated to 1500 words')
        except Exception as e:
            print(f'  failed on union_id={row["union_id"]}: {e}')
            mean_s, n_tok = float('nan'), 0
    except Exception as e:
        print(f'  failed on union_id={row["union_id"]}: {e}')
        mean_s, n_tok = float('nan'), 0
    records.append({
        'union_id': int(row['union_id']),
        'mean_surprisal': mean_s,
        'n_tokens': n_tok,
        'model_name': MODEL_NAME,
    })
    since_flush += 1
    if since_flush >= CHECKPOINT_EVERY:
        pd.DataFrame(records).to_parquet(PATH_SURPRISAL, index=False)
        since_flush = 0
        pbar.set_postfix(flushed=len(records))

# Final flush
pd.DataFrame(records).to_parquet(PATH_SURPRISAL, index=False)
print(f'\nPhase B complete. Saved {len(records):,} per-film surprisal scores.')

surprisal_df = pd.read_parquet(PATH_SURPRISAL)
print(f'  mean surprisal distribution: '
      f'p50={surprisal_df["mean_surprisal"].median():.3f}  '
      f'p25={surprisal_df["mean_surprisal"].quantile(0.25):.3f}  '
      f'p75={surprisal_df["mean_surprisal"].quantile(0.75):.3f}')
print(f'  failed (NaN) films         : {surprisal_df["mean_surprisal"].isna().sum()}')

No checkpoint found; starting from scratch.
45,958 films to process


surprisal:   0%|          | 0/45958 [00:00<?, ?it/s]


Phase B complete. Saved 45,958 per-film surprisal scores.
  mean surprisal distribution: p50=2.696  p25=2.476  p75=2.962
  failed (NaN) films         : 0


In [5]:
# ----------------------------------------------------------------------
# Phase C — Per-film mean stockiness
#
# For each film f tagged with genres G_f:
#
#   mean_stockiness(f) = mean over (e in f's elements, g in G_f) of
#                        ensemble(e, g)
#
# where the element set is filtered to is_narrative_element=True so
# the measure reflects narrative stockiness, not extraction noise.
#
# Two intermediate computations:
#   1. Build the per-(film, element) join, including the film's genres
#      so the genre dimension is materialized in the join.
#   2. Look up ensemble(e, g) for each (film, element, genre) tuple
#      from stockiness_long_classified.
# ----------------------------------------------------------------------

if PATH_MEAN_STOCK.exists():
    print(f'Loading existing Phase C checkpoint: {PATH_MEAN_STOCK}')
    mean_stock_df = pd.read_parquet(PATH_MEAN_STOCK)
else:
    print('Running Phase C: computing per-film mean stockiness...')

    print('  loading element index and enriched stockiness table...')
    ei = pd.read_parquet(PATH_ELEM_IDX)
    stk = pd.read_parquet(PATH_ENRICHED)

    # Filter the stockiness table to narrative elements only.
    stk_narr = stk[stk['is_narrative_element']].copy()
    print(f'  {len(stk_narr):,} narrative (element, type, genre) cells')

    # Drop duplicate (union_id, element, element_type) tuples in the
    # element index — a film mentioning an element five times contributes
    # once to the mean.
    ei_dedup = ei.drop_duplicates(['union_id','element','element_type']).reset_index(drop=True)
    print(f'  {len(ei_dedup):,} unique (film, element) memberships')

    # Per-film genre list (one row per film, with `genres_canonical` as a list).
    film_genres = movies[['union_id','genres_canonical']].copy()
    film_genres['genres_canonical'] = film_genres['genres_canonical'].apply(
        lambda gs: [g for g in gs if g in CANONICAL_GENRES]
    )

    # Materialize one row per (film, element, genre) where the genre is in the
    # film's tag list. Use explode for memory efficiency.
    print('  exploding to per-(film, element, genre) rows...')
    film_genres_long = film_genres.explode('genres_canonical').rename(
        columns={'genres_canonical': 'genre'}).dropna(subset=['genre'])
    fe_g = ei_dedup.merge(film_genres_long, on='union_id', how='inner')

    # Now join with the narrative stockiness rows.
    print('  joining with narrative stockiness...')
    joined = fe_g.merge(
        stk_narr[['element','element_type','genre','ensemble']],
        on=['element','element_type','genre'],
        how='inner',
    )
    print(f'  {len(joined):,} (film, element, genre) cells with narrative-stockiness scores')

    # Mean per (film, genre): inner mean over elements in that film and genre.
    per_film_genre = (joined.groupby(['union_id','genre'])['ensemble']
                            .mean()
                            .reset_index(name='mean_stockiness_in_genre'))

    # Overall per-film mean: outer mean across the film's genres.
    per_film = (per_film_genre.groupby('union_id')['mean_stockiness_in_genre']
                              .mean()
                              .reset_index(name='mean_stockiness_overall'))

    # Counts: number of (element, genre) cells that contributed.
    per_film_counts = joined.groupby('union_id').size().reset_index(name='n_element_genre_cells')
    per_film_n_elements = (joined.groupby('union_id')['element'].nunique()
                                 .reset_index(name='n_narrative_elements'))

    mean_stock_df = per_film.merge(per_film_counts, on='union_id', how='left')
    mean_stock_df = mean_stock_df.merge(per_film_n_elements, on='union_id', how='left')

    # Save per-(film, genre) means too — useful for the per-genre correlation.
    per_film_genre.to_parquet(OUT_DIR / 'per_film_genre_mean_stockiness.parquet', index=False)

    mean_stock_df.to_parquet(PATH_MEAN_STOCK, index=False)
    print(f'Saved: {PATH_MEAN_STOCK}')

print(f'\nPhase C complete.')
print(f'  films with a mean stockiness score: {len(mean_stock_df):,}')
print(f'  mean stockiness distribution: '
      f'p50={mean_stock_df["mean_stockiness_overall"].median():.3f}  '
      f'p25={mean_stock_df["mean_stockiness_overall"].quantile(0.25):.3f}  '
      f'p75={mean_stock_df["mean_stockiness_overall"].quantile(0.75):.3f}')
print(f'  median narrative elements per film: '
      f'{int(mean_stock_df["n_narrative_elements"].median())}')

Running Phase C: computing per-film mean stockiness...
  loading element index and enriched stockiness table...
  319,542 narrative (element, type, genre) cells
  4,305,936 unique (film, element) memberships
  exploding to per-(film, element, genre) rows...
  joining with narrative stockiness...
  5,952,180 (film, element, genre) cells with narrative-stockiness scores
Saved: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/04_surprisal/per_film_mean_stockiness.parquet

Phase C complete.
  films with a mean stockiness score: 45,953
  mean stockiness distribution: p50=2.736  p25=2.089  p75=3.651
  median narrative elements per film: 39


In [6]:
# ----------------------------------------------------------------------
# Phase D — Correlation analysis
#
# We compute TWO correlations per genre and corpus-wide:
#
#   1. RAW Pearson r between mean_surprisal and mean_stockiness_overall.
#      This is the simple bivariate number.
#
#   2. PARTIAL Pearson r controlling for log(word_count). This is the
#      headline number for the paper, because both measures are
#      confounded by plot length in opposite directions:
#        - Shorter plots have higher surprisal (less context for the LM)
#        - Shorter plots have higher mean stockiness (fewer elements,
#          less dilution toward the mean)
#      The two length confounds make the raw correlation close to zero
#      even when a real signal exists. Controlling for length surfaces
#      the underlying relationship.
#
# Spearman ρ is reported alongside as a robustness check on the
# linearity assumption.
#
# The doctrinal discriminant-validity claim is satisfied if |partial r|
# is well below 0.40. The expected direction of the partial r is
# NEGATIVE: predictable plots use stock elements, unpredictable plots
# use less-stock elements.
# ----------------------------------------------------------------------
from scipy.stats import pearsonr, spearmanr

def partial_pearson(x, y, controls):
    """Partial Pearson r of x and y controlling for the columns of
    `controls`. Implementation: regress out controls from each via OLS,
    then correlate the residuals. Returns (r, p)."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    C = np.asarray(controls, dtype=float)
    if C.ndim == 1:
        C = C.reshape(-1, 1)
    # Add intercept column
    X = np.column_stack([np.ones(len(x)), C])
    # Residualize x and y against X
    bx, *_ = np.linalg.lstsq(X, x, rcond=None)
    by, *_ = np.linalg.lstsq(X, y, rcond=None)
    rx = x - X @ bx
    ry = y - X @ by
    return pearsonr(rx, ry)

# Join surprisal, mean stockiness, and movie metadata (we need word_count
# as the length control).
joined = (surprisal_df
          .merge(mean_stock_df, on='union_id', how='inner')
          .merge(movies[['union_id', 'word_count']], on='union_id', how='left')
          .dropna(subset=['mean_surprisal','mean_stockiness_overall','word_count']))
joined['log_word_count'] = np.log(joined['word_count'].clip(lower=1))
print(f'Films with surprisal + stockiness + word_count: {len(joined):,}')

# Corpus-wide: raw and partial
r_pearson, p_pearson = pearsonr(joined['mean_surprisal'], joined['mean_stockiness_overall'])
r_spearman, p_spearman = spearmanr(joined['mean_surprisal'], joined['mean_stockiness_overall'])
r_partial, p_partial = partial_pearson(
    joined['mean_surprisal'], joined['mean_stockiness_overall'],
    joined['log_word_count'].values,
)

print(f'\nCorpus-wide:')
print(f'  raw Pearson r        = {r_pearson:+.3f}  (p = {p_pearson:.3e})')
print(f'  raw Spearman ρ       = {r_spearman:+.3f}  (p = {p_spearman:.3e})')
print(f'  PARTIAL Pearson r    = {r_partial:+.3f}  (p = {p_partial:.3e})   ← headline (controls for log(word_count))')
print(f'  partial R²           = {r_partial**2*100:.2f}%  (variance shared after removing length confound)')

# Per-genre: raw and partial
per_film_genre = pd.read_parquet(OUT_DIR / 'per_film_genre_mean_stockiness.parquet')
per_film_genre = (per_film_genre
                  .merge(surprisal_df[['union_id','mean_surprisal']], on='union_id')
                  .merge(movies[['union_id','word_count']], on='union_id', how='left')
                  .dropna(subset=['mean_surprisal','mean_stockiness_in_genre','word_count']))
per_film_genre['log_word_count'] = np.log(per_film_genre['word_count'].clip(lower=1))

rows = []
for g in CANONICAL_GENRES:
    sub = per_film_genre[per_film_genre['genre']==g]
    if len(sub) < 30:
        continue
    rp, pp = pearsonr(sub['mean_surprisal'], sub['mean_stockiness_in_genre'])
    rs, ps = spearmanr(sub['mean_surprisal'], sub['mean_stockiness_in_genre'])
    rpartial, ppartial = partial_pearson(
        sub['mean_surprisal'], sub['mean_stockiness_in_genre'],
        sub['log_word_count'].values,
    )
    rows.append({
        'genre': g, 'n_films': len(sub),
        'raw_pearson_r': rp, 'raw_pearson_p': pp,
        'raw_spearman_rho': rs, 'raw_spearman_p': ps,
        'partial_pearson_r': rpartial, 'partial_pearson_p': ppartial,
    })

corr_per_genre = pd.DataFrame(rows).sort_values('partial_pearson_r')
print(f'\nPer-genre (sorted ascending by partial r — most-negative first):')
print(f'  {"genre":<18} {"n_films":>8} {"raw r":>8} {"partial r":>11} {"raw rho":>9} {"direction":>20}')
for _, r in corr_per_genre.iterrows():
    pr = r['partial_pearson_r']
    direction = ('strongly intuitive (neg)' if pr <= -0.15
                 else 'modestly intuitive (neg)' if pr <= -0.05
                 else 'flat' if abs(pr) < 0.05
                 else 'counter-intuitive (pos)')
    print(f'  {r["genre"]:<18} {int(r["n_films"]):>8} '
          f'{r["raw_pearson_r"]:>+8.3f} {r["partial_pearson_r"]:>+11.3f} '
          f'{r["raw_spearman_rho"]:>+9.3f}  {direction:>22}')

# Save corpus + per-genre rows
corpus_row = pd.DataFrame([{
    'genre': 'CORPUS', 'n_films': len(joined),
    'raw_pearson_r': r_pearson, 'raw_pearson_p': p_pearson,
    'raw_spearman_rho': r_spearman, 'raw_spearman_p': p_spearman,
    'partial_pearson_r': r_partial, 'partial_pearson_p': p_partial,
}])
corr_table = pd.concat([corpus_row, corr_per_genre], ignore_index=True)
corr_table.to_parquet(PATH_CORR_TBL, index=False)
print(f'\nSaved: {PATH_CORR_TBL}')

Films with surprisal + stockiness + word_count: 45,953

Corpus-wide:
  raw Pearson r        = +0.004  (p = 4.126e-01)
  raw Spearman ρ       = -0.026  (p = 4.408e-08)
  PARTIAL Pearson r    = -0.164  (p = 6.921e-273)   ← headline (controls for log(word_count))
  partial R²           = 2.67%  (variance shared after removing length confound)

Per-genre (sorted ascending by partial r — most-negative first):
  genre               n_films    raw r   partial r   raw rho            direction
  Fantasy                2429   -0.097      -0.234    -0.140  strongly intuitive (neg)
  History                2882   -0.061      -0.186    -0.095  strongly intuitive (neg)
  Romance                8219   +0.049      -0.181    -0.013  strongly intuitive (neg)
  Drama                 22363   +0.004      -0.169    -0.022  strongly intuitive (neg)
  Horror                 4363   -0.046      -0.165    -0.069  strongly intuitive (neg)
  Comedy                14720   -0.000      -0.149    -0.027  modestly intu

In [7]:
# ----------------------------------------------------------------------
# Write the human-readable report.
#
# Headline: partial Pearson r controlling for log(word_count).
# Raw Pearson r and Spearman ρ reported alongside for transparency.
# ----------------------------------------------------------------------
lines = []
def out(line=''):
    print(line); lines.append(line)

out('=' * 70)
out('NOTEBOOK 04: PLOT-LEVEL SURPRISAL — DISCRIMINANT VALIDITY REPORT')
out('=' * 70)
out(f'language model        : {MODEL_NAME}')
out(f'films with surprisal  : {(~surprisal_df["mean_surprisal"].isna()).sum():,}')
out(f'films with stockiness : {len(mean_stock_df):,}')
out(f'films in both + length: {len(joined):,}')
out('')
out('SURPRISAL DISTRIBUTION (nats per token, full corpus)')
out(f'  p10    : {surprisal_df["mean_surprisal"].quantile(0.10):.3f}')
out(f'  p25    : {surprisal_df["mean_surprisal"].quantile(0.25):.3f}')
out(f'  median : {surprisal_df["mean_surprisal"].median():.3f}')
out(f'  p75    : {surprisal_df["mean_surprisal"].quantile(0.75):.3f}')
out(f'  p90    : {surprisal_df["mean_surprisal"].quantile(0.90):.3f}')
out('')
out('PER-FILM MEAN STOCKINESS DISTRIBUTION')
out(f'  p10    : {mean_stock_df["mean_stockiness_overall"].quantile(0.10):.3f}')
out(f'  p25    : {mean_stock_df["mean_stockiness_overall"].quantile(0.25):.3f}')
out(f'  median : {mean_stock_df["mean_stockiness_overall"].median():.3f}')
out(f'  p75    : {mean_stock_df["mean_stockiness_overall"].quantile(0.75):.3f}')
out(f'  p90    : {mean_stock_df["mean_stockiness_overall"].quantile(0.90):.3f}')
out('')

out('=' * 70)
out('LENGTH CONFOUND CHECK')
out('=' * 70)
r_surp_len = pearsonr(joined['mean_surprisal'], joined['log_word_count'])[0]
r_stock_len = pearsonr(joined['mean_stockiness_overall'], joined['log_word_count'])[0]
out(f'  Pearson r(surprisal, log_word_count) = {r_surp_len:+.3f}')
out(f'  Pearson r(stockiness, log_word_count) = {r_stock_len:+.3f}')
out('  Both measures are correlated with plot length in the same direction,')
out('  which makes the RAW correlation between surprisal and stockiness')
out('  close to zero even when a real underlying signal exists. The headline')
out('  number is the PARTIAL correlation, which removes the length confound.')
out('')

out('=' * 70)
out('HEADLINE: corpus-wide Pearson r between mean surprisal and mean per-film')
out('narrative stockiness, with and without length control')
out('=' * 70)
out(f'  RAW     Pearson r   = {r_pearson:+.3f}  (n = {len(joined):,}, p = {p_pearson:.3e})')
out(f'  PARTIAL Pearson r   = {r_partial:+.3f}  (controlling for log(word_count), p = {p_partial:.3e})')
out(f'  Spearman ρ (raw)    = {r_spearman:+.3f}')
out(f'  partial R²          = {r_partial**2*100:.2f}%  (variance shared after removing length confound)')
out('')
out('Interpretation guide (use the PARTIAL r for the doctrinal claim):')
out('  |partial r| < 0.20  weak / no relationship — element-level measures fully independent of plot predictability')
out('  |partial r| 0.20-0.40  modest correlation — directional agreement, framework retains independent doctrinal claim')
out('  |partial r| 0.40-0.60  moderate correlation — framework shares meaningful variance with plot predictability but is not redundant')
out('  |partial r| > 0.60  strong correlation — element-level measures are essentially a re-skinned plot-predictability metric (doctrinal claim weakened)')
out('')
out('Expected sign of the partial r: NEGATIVE')
out('  Negative partial r means predictable plots use stock elements and unpredictable')
out('  plots use less-stock elements, which is the doctrinally intuitive direction.')
out('  A positive partial r would be counter-intuitive and worth investigating.')
out('')

out('=' * 70)
out('PER-GENRE CORRELATIONS (sorted ascending by partial r)')
out('=' * 70)
out(f'  {"genre":<18} {"n_films":>8} {"raw r":>8} {"partial r":>11} {"raw rho":>9}')
for _, r in corr_per_genre.iterrows():
    out(f'  {r["genre"]:<18} {int(r["n_films"]):>8} '
        f'{r["raw_pearson_r"]:>+8.3f} {r["partial_pearson_r"]:>+11.3f} '
        f'{r["raw_spearman_rho"]:>+9.3f}')
out('')

out('=' * 70)
out('KNOWN CONFOUND: HIGH-SURPRISAL TAIL')
out('=' * 70)
q95 = joined['mean_surprisal'].quantile(0.95)
q05 = joined['mean_surprisal'].quantile(0.05)
top5 = joined[joined['mean_surprisal'] > q95]
bot5 = joined[joined['mean_surprisal'] < q05]
out(f'  top 5% by surprisal (n={len(top5):,}): median word_count = {int(top5["word_count"].median())}')
out(f'  bot 5% by surprisal (n={len(bot5):,}): median word_count = {int(bot5["word_count"].median())}')
out('  Inspection of the high-surprisal tail shows it is dominated by short plot')
out('  summaries of foreign-language films awkwardly translated into English. The')
out('  surprisal is high because the language is non-native, not because the')
out('  narrative is unusual. This is a corpus-quality artifact; the partial-')
out('  correlation framing accommodates it because length partly proxies for')
out('  text quality. A paper-time robustness check could filter to clean-English')
out('  plots only.')

PATH_REPORT.write_text('\n'.join(lines))
print(f'\nReport written to {PATH_REPORT}')

NOTEBOOK 04: PLOT-LEVEL SURPRISAL — DISCRIMINANT VALIDITY REPORT
language model        : Qwen/Qwen2.5-7B
films with surprisal  : 45,958
films with stockiness : 45,953
films in both + length: 45,953

SURPRISAL DISTRIBUTION (nats per token, full corpus)
  p10    : 2.290
  p25    : 2.476
  median : 2.696
  p75    : 2.962
  p90    : 3.249

PER-FILM MEAN STOCKINESS DISTRIBUTION
  p10    : 1.596
  p25    : 2.089
  median : 2.736
  p75    : 3.651
  p90    : 4.896

LENGTH CONFOUND CHECK
  Pearson r(surprisal, log_word_count) = -0.522
  Pearson r(stockiness, log_word_count) = -0.265
  Both measures are correlated with plot length in the same direction,
  which makes the RAW correlation between surprisal and stockiness
  close to zero even when a real underlying signal exists. The headline
  number is the PARTIAL correlation, which removes the length confound.

HEADLINE: corpus-wide Pearson r between mean surprisal and mean per-film
narrative stockiness, with and without length control
  RAW    

## Outputs produced

- **`per_film_surprisal.parquet`** — one row per film: `union_id`, `mean_surprisal`, `n_tokens`, `model_name`. The raw surprisal numbers, idempotently produced.
- **`per_film_mean_stockiness.parquet`** — one row per film: `union_id`, `mean_stockiness_overall`, `n_element_genre_cells`, `n_narrative_elements`. The doubly-averaged stockiness measure used in the headline correlation.
- **`per_film_genre_mean_stockiness.parquet`** — one row per (film, genre): `mean_stockiness_in_genre`. Used for the per-genre correlation.
- **`correlation_table.parquet`** — corpus-wide row plus one row per genre: `n_films`, `pearson_r`, `pearson_p`, `spearman_rho`, `spearman_p`.
- **`surprisal_report.txt`** — headline Pearson r and Spearman ρ, surprisal and stockiness distributions, per-genre correlation table.

## How to read the headline

The doctrinal interpretation hinges on the magnitude of the Pearson r at the corpus level.

- `|r| < 0.40` is the expected and desired range. The element-level stockiness measures correlate directionally with plot surprisal (more-conventional plots contain more-stock elements) but the relationship is loose enough that the framework retains a doctrinal claim that the measures capture element-level originality independently of running-text predictability.
- `|r| > 0.60` is the failure mode. The element-level measures are essentially a re-skinned plot-predictability metric and the doctrinal claim that they operationalize the doctrine independently of plot-level surprisal is weakened.

Per-genre correlations are reported alongside the corpus-wide number. Substantial heterogeneity across genres (some genres with strong correlation, others with weak) is doctrinally interesting and worth discussing in the paper; it suggests that the relationship between conventional vocabulary and predictable prose is itself genre-dependent.

## Sensitivity check

If the headline result is borderline (Pearson r between 0.40 and 0.50), the paper should run the same notebook against **Llama-3.1-8B** as a second long-context model and report both correlations. A consistent finding across two architecturally different models strengthens the discriminant-validity claim. The notebook supports this swap with a single constant change (the `MODEL_NAME` line in Cell 1).

## Known limitations

- The plot summaries are Wikipedia synopses, not original screenplays or shooting scripts. Surprisal is measured on derivative narrative summarizations, not on the source narratives themselves. This is a corpus-design choice imposed by data availability; the doctrinal claim translates because plot summaries preserve element-level structure that doctrine cares about.
- The corpus contains plot summaries written across decades. Wikipedia editorial conventions have shifted; older articles tend to be denser and more terse than newer ones. Surprisal correlates weakly with plot length; a stratified analysis by decade would surface any cohort effect.
- Qwen2.5-7B was trained on a mixed corpus that includes Wikipedia text. The model has seen the training set; its surprisal on plot summaries is therefore not zero-shot. The discriminant-validity claim survives this caveat because the comparison is *relative* — between films within the same corpus, not between this corpus and held-out text.
